In [1]:
# Cell 1 — imports
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv('../../.env')

from src.utils.config import settings
from src.utils.logger import get_logger
log = get_logger("setup_check")

log.info("✅ Imports working")

# Cell 2 — load all datasets
files = {
    'movies_metadata': '../../data/raw/movies_metadata.csv',
    'credits':         '../../data/raw/credits.csv',
    'keywords':        '../../data/raw/keywords.csv',
    'links':           '../../data/raw/links.csv',
    'links_small':     '../../data/raw/links_small.csv',
    'ratings_small':   '../../data/raw/ratings_small.csv',
}

dfs = {}
for name, path in files.items():
    df = pd.read_csv(path, low_memory=False)
    dfs[name] = df
    print(f"✅ {name:25s} {df.shape[0]:>8,} rows × {df.shape[1]:>3} cols")

# Cell 3 — check services
import redis, psycopg2
from qdrant_client import QdrantClient
import mlflow

# Redis
r = redis.Redis(host=settings.REDIS_HOST, port=settings.REDIS_PORT)
print(f"Redis:    {'✅ connected' if r.ping() else '❌ failed'}")

# Qdrant
qc = QdrantClient(host=settings.QDRANT_HOST, port=settings.QDRANT_PORT)
print(f"Qdrant:   ✅ connected — {len(qc.get_collections().collections)} collections")

# MLflow
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
exp = mlflow.set_experiment(settings.MLFLOW_EXPERIMENT_NAME)
print(f"MLflow:   ✅ experiment '{exp.name}' ready")

# PostgreSQL
try:
    conn = psycopg2.connect(
        host=settings.POSTGRES_HOST, port=settings.POSTGRES_PORT,
        dbname=settings.POSTGRES_DB, user=settings.POSTGRES_USER,
        password=settings.POSTGRES_PASSWORD
    )
    print("Postgres: ✅ connected")
    conn.close()
except Exception as e:
    print(f"Postgres: ❌ {e}")

2026-06-01 22:36:16 | INFO | setup_check | ✅ Imports working
✅ movies_metadata             45,466 rows ×  24 cols
✅ credits                     45,476 rows ×   3 cols
✅ keywords                    46,419 rows ×   2 cols
✅ links                       45,843 rows ×   3 cols
✅ links_small                  9,125 rows ×   3 cols
✅ ratings_small              100,004 rows ×   4 cols
Redis:    ✅ connected
Qdrant:   ✅ connected — 0 collections
MLflow:   ✅ experiment 'recsys_experiments' ready
Postgres: ❌ connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "recsys_user"

